# Лабораторная работа №1 — Одномерный поиск и градиентный спуск
## 6413 Круглов, Косцов
### Вариант №21


---

## Часть A. Одномерный поиск

### Задание A1. Целевая функция
- **Целевая функция (вариант №21):**
$$f(x) = x + \frac{1}{x - 2.5}$$
- **Отрезок поиска:** $[3, 6]$

Проверим унимодальность аналитически: на отрезке $[3, 6]$ знаменатель $x - 2.5 > 0$, вторая производная
$$f''(x) = \frac{2}{(x - 2.5)^3} > 0,$$
значит, функция строго выпукла на всём отрезке и имеет единственный минимум:
$$f'(x) = 1 - \frac{1}{(x - 2.5)^2} = 0 \implies x - 2.5 = 1 \implies x^* = 3.5, \quad f(x^*) = 4.5.$$

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DELTA = 1e-4
EPSILON = DELTA * 0.1  # 1e-5, малая величина по сравнению с 2*DELTA

def f(x):
    return x + 1.0 / (x - 2.5)

a0, b0 = 3.0, 6.0
x_exact = 3.5
f_exact = f(x_exact)

x_grid = np.linspace(a0, b0, 1000)
plt.figure(figsize=(9, 4.8))
plt.plot(x_grid, f(x_grid), lw=2, label=r'$f(x) = x + \frac{1}{x - 2.5}$')
plt.axvline(x_exact, color='red', linestyle='--', alpha=0.7, label=r'Минимум $x^* = 3.5$')
plt.scatter([x_exact], [f_exact], color='red', zorder=5)
plt.xlabel('x')
plt.ylabel('f(x)')
plt.title('Целевая функция варианта №21 на отрезке [3, 6]')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()

print(f"Аналитический минимум: x* = {x_exact:.6f}, f(x*) = {f_exact:.6f}")

### Задание A2. Метод дихотомии
Выбираем параметр $\varepsilon = 0.1\Delta = 10^{-5} \ll 2\Delta$.

In [ ]:
def dichotomy(f, a, b, delta, epsilon):
    assert 0 < epsilon < 2 * delta, "epsilon должен быть положительным и меньше 2*delta"
    iterations = 0
    evaluations = 0
    history = [{
        "iterations": 0,
        "evaluations": 0,
        "a": a,
        "b": b,
        "length": b - a,
        "x1": None,
        "x2": None,
    }]

    while b - a > 2 * delta:
        x1 = (a + b - epsilon) / 2.0
        x2 = (a + b + epsilon) / 2.0
        f1 = f(x1)
        f2 = f(x2)
        evaluations += 2
        iterations += 1

        if f1 < f2:
            b = x2
        else:
            a = x1

        history.append({
            "iterations": iterations,
            "evaluations": evaluations,
            "a": a,
            "b": b,
            "length": b - a,
            "x1": x1,
            "x2": x2,
        })

    x_min = (a + b) / 2.0
    return {
        "method": "Дихотомия",
        "x_min": x_min,
        "f_min": f(x_min),
        "interval_length": b - a,
        "achieved_delta": (b - a) / 2.0,
        "iterations": iterations,
        "evaluations": evaluations,
        "interval": (a, b),
        "history": history,
    }

result_d = dichotomy(f, a0, b0, DELTA, EPSILON)
for k in ["x_min", "f_min", "interval_length", "iterations", "evaluations", "interval"]:
    print(f"{k}: {result_d[k]}")

### Задание A3. Метод золотого сечения

In [ ]:
def golden_section(f, a, b, delta):
    tau = (math.sqrt(5) - 1) / 2.0
    history = [{
        "iterations": 0,
        "evaluations": 0,
        "a": a,
        "b": b,
        "length": b - a,
        "x1": None,
        "x2": None,
    }]

    if b - a <= 2 * delta:
        x_min = (a + b) / 2.0
        return {
            "method": "Золотое сечение",
            "x_min": x_min,
            "f_min": f(x_min),
            "interval_length": b - a,
            "achieved_delta": (b - a) / 2.0,
            "iterations": 0,
            "evaluations": 0,
            "interval": (a, b),
            "history": history,
        }

    x1 = b - tau * (b - a)
    x2 = a + tau * (b - a)
    f1 = f(x1)
    f2 = f(x2)
    evaluations = 2
    iterations = 0

    while True:
        iterations += 1
        cur_x1, cur_x2 = x1, x2
        if f1 < f2:
            b = x2
            x2 = x1
            f2 = f1
            history.append({"iterations": iterations, "evaluations": evaluations, "a": a, "b": b, "length": b - a, "x1": cur_x1, "x2": cur_x2})
            if b - a <= 2 * delta:
                break
            x1 = b - tau * (b - a)
            f1 = f(x1)
            evaluations += 1
        else:
            a = x1
            x1 = x2
            f1 = f2
            history.append({"iterations": iterations, "evaluations": evaluations, "a": a, "b": b, "length": b - a, "x1": cur_x1, "x2": cur_x2})
            if b - a <= 2 * delta:
                break
            x2 = a + tau * (b - a)
            f2 = f(x2)
            evaluations += 1

    x_min = (a + b) / 2.0
    return {
        "method": "Золотое сечение",
        "x_min": x_min,
        "f_min": f(x_min),
        "interval_length": b - a,
        "achieved_delta": (b - a) / 2.0,
        "iterations": iterations,
        "evaluations": evaluations,
        "interval": (a, b),
        "history": history,
    }

result_g = golden_section(f, a0, b0, DELTA)
for k in ["x_min", "f_min", "interval_length", "iterations", "evaluations", "interval"]:
    print(f"{k}: {result_g[k]}")

### Задание A4. Первое сравнение

In [ ]:
comparison_dg = pd.DataFrame([
    {
        "Метод": r["method"],
        "x*": r["x_min"],
        "f(x*)": r["f_min"],
        "Итоговая длина интервала": r["interval_length"],
        "Итераций": r["iterations"],
        "Вычислений f(x)": r["evaluations"],
    }
    for r in [result_d, result_g]
])
display(comparison_dg.round(8))

### Задание A5. Метод Фибоначчи

In [ ]:
def fibonacci_search(f, a, b, budget):
    n = budget
    assert n >= 2, "Бюджет вычислений должен быть >= 2"
    fib = [0, 1]
    for _ in range(2, n + 2):
        fib.append(fib[-1] + fib[-2])

    x1 = a + (fib[n - 1] / fib[n + 1]) * (b - a)
    x2 = a + (fib[n] / fib[n + 1]) * (b - a)
    f1 = f(x1)
    f2 = f(x2)
    evaluations = 2
    iterations = 0

    history = [{
        "iterations": 0,
        "evaluations": 0,
        "a": a,
        "b": b,
        "length": b - a,
        "x1": None,
        "x2": None,
    }]

    for k in range(1, n - 1):
        iterations += 1
        cur_x1, cur_x2 = x1, x2
        if f1 < f2:
            b = x2
            x2 = x1
            f2 = f1
            x1 = a + (fib[n - k - 1] / fib[n - k + 1]) * (b - a)
            if k < n - 2:
                f1 = f(x1)
                evaluations += 1
        else:
            a = x1
            x1 = x2
            f1 = f2
            x2 = a + (fib[n - k] / fib[n - k + 1]) * (b - a)
            if k < n - 2:
                f2 = f(x2)
                evaluations += 1

        history.append({
            "iterations": iterations,
            "evaluations": evaluations,
            "a": a,
            "b": b,
            "length": b - a,
            "x1": cur_x1,
            "x2": cur_x2,
        })

    # Заключительный шаг: сдвиг на малую величину eps для последнего вычисления f
    eps = (b - a) * 0.01
    x2 = x1 + eps
    f2 = f(x2)
    evaluations += 1
    iterations += 1

    if f1 < f2:
        b = x2
    else:
        a = x1

    history.append({
        "iterations": iterations,
        "evaluations": evaluations,
        "a": a,
        "b": b,
        "length": b - a,
        "x1": x1,
        "x2": x2,
    })

    x_min = (a + b) / 2.0
    return {
        "method": "Фибоначчи",
        "x_min": x_min,
        "f_min": f(x_min),
        "interval_length": b - a,
        "achieved_delta": (b - a) / 2.0,
        "iterations": iterations,
        "evaluations": evaluations,
        "interval": (a, b),
        "history": history,
    }

budget = result_g["evaluations"]
result_fib = fibonacci_search(f, a0, b0, budget)
print(f"Бюджет вычислений (как у золотого сечения): {budget}")
for k in ["x_min", "f_min", "interval_length", "iterations", "evaluations", "interval"]:
    print(f"{k}: {result_fib[k]}")

### Задание A6. Итоговое сравнение

In [ ]:
results = [result_d, result_g, result_fib]

comparison_all = pd.DataFrame([
    {
        "Метод": r["method"],
        "x*": r["x_min"],
        "f(x*)": r["f_min"],
        "Итоговая длина интервала": r["interval_length"],
        "Точность (b - a)/2": r["achieved_delta"],
        "Итераций": r["iterations"],
        "Вычислений f(x)": r["evaluations"],
    }
    for r in results
])
display(comparison_all.round(8))

plt.figure(figsize=(10, 5.5))
for r in results:
    evals = [h["evaluations"] for h in r["history"]]
    lengths = [h["length"] for h in r["history"]]
    plt.plot(evals, lengths, marker="o", markersize=4, label=r["method"])

plt.axhline(2 * DELTA, color="black", linestyle="--", label=r'$2\Delta$')
plt.yscale("log")
plt.xlabel("Количество вычислений f(x)")
plt.ylabel("Длина текущего интервала (лог. шкала)")
plt.title(r"Сравнение сходимости методов одномерного поиска ($\Delta = 10^{-4}$)")
plt.grid(True, which="both", linestyle="--", alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()

#### Ответы на вопросы (Часть A)
1. **Какой метод потребовал меньше вычислений функции при заданной точности: дихотомия или золотое сечение?**  
   Золотое сечение: при $\Delta = 10^{-4}$ оно потребовало **21** вычисление $f(x)$, тогда как дихотомия — **28** вычислений.
2. **Какой итоговый интервал получился у метода Фибоначчи при том же бюджете, что у золотого сечения?**  
   При бюджете $N = 21$ длина итогового интервала у метода Фибоначчи составила `1.7277e-04` (точность `8.6387e-05`), тогда как у золотого сечения — `1.9832e-04` (точность `9.9160e-05`).
3. **Всегда ли экспериментальное преимущество одного метода одинаково заметно?**  
   Нет: при грубой точности (большом $\Delta$) и коротком поиске разница в 1–2 вычисления малозаметна, а фактическая ошибка $|x^* - x_{\text{exact}}|$ может случайно оказаться меньше у любого метода в зависимости от попадания границ.
4. **Как изменение $\Delta$ влияет на разницу между методами?**  
   При уменьшении $\Delta$ число шагов растёт логарифмически, и преимущество золотого сечения над дихотомией по числу вычислений $f(x)$ (примерно на 25–28%) становится всё более ощутимым в абсолютных числах.
5. **Почему количество итераций и количество вычислений целевой функции — разные характеристики?**  
   В дихотомии каждая итерация требует **2 новых вычислений** функции, а в золотом сечении и методе Фибоначчи (после первого шага) — только **1 новое вычисление**.
6. **Почему при сравнении методов важно учитывать именно количество вычислений $f(x)$?**  
   В практических задачах оптимизации вычисление целевой функции является самой дорогой операцией (например, запуск симуляции или оценка модели на данных), поэтому именно число обращений к $f(x)$ определяет общую вычислительную стоимость.

---
## Часть B. Градиентный спуск

Рассмотрим функцию:
$$f(x) = (x - 3)^2 + 2.$$

### Задание B1. Функция и производная
- Аналитическая производная: $f'(x) = 2(x - 3)$.
- Аналитический минимум: $f'(x) = 0 \implies x^* = 3$, $f(x^*) = 2$.

In [ ]:
def objective(x):
    return (x - 3.0) ** 2 + 2.0

def derivative(x):
    return 2.0 * (x - 3.0)

def gradient_descent(x_init, lr, n_iter, func, grad_func):
    x = float(x_init)
    x_hist = [x]
    f_hist = [func(x)]
    for _ in range(n_iter):
        x = x - lr * grad_func(x)
        x_hist.append(x)
        f_hist.append(func(x))
    return np.array(x_hist), np.array(f_hist)

### Задание B3. Влияние learning rate

In [ ]:
learning_rates = [0.01, 0.1, 0.9, 1.1]
x_start = 0.0
n_steps = 25

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for lr in learning_rates:
    xh, fh = gradient_descent(x_start, lr, n_steps, objective, derivative)
    axes[0].plot(range(n_steps + 1), fh, marker='o', markersize=3, label=f'η = {lr}')
    axes[1].plot(range(n_steps + 1), xh, marker='o', markersize=3, label=f'η = {lr}')
    print(f"η = {lr:>4}: финальное x = {xh[-1]:.6f}, f(x) = {fh[-1]:.6f}")

axes[0].set_ylim(0, 30)
axes[0].set_xlabel('Итерация t')
axes[0].set_ylabel(r'$f(x_t)$')
axes[0].set_title(r'Зависимость $f(x_t)$ от номера итерации')
axes[0].grid(True, linestyle='--', alpha=0.5)
axes[0].legend()

axes[1].axhline(3.0, color='black', linestyle=':', label=r'$x^* = 3$')
axes[1].set_ylim(-6, 12)
axes[1].set_xlabel('Итерация t')
axes[1].set_ylabel(r'$x_t$')
axes[1].set_title(r'Траектория координаты $x_t$')
axes[1].grid(True, linestyle='--', alpha=0.5)
axes[1].legend()

plt.tight_layout()
plt.show()

#### Ответы на вопросы (Часть B)
1. **При каком шаге алгоритм сходится быстрее?**  
   Среди рассмотренных значений быстрее всего сходится при $\eta = 0.1$ (и $\eta = 0.9$, хотя при $0.9$ с колебаниями); теоретически оптимальный шаг для $f(x) = (x-3)^2+2$ ($f''=2$) равен $\eta = 1/f'' = 0.5$ — сходимость за 1 шаг.
2. **Когда возникают колебания?**  
   При $\eta > 0.5$ (в эксперименте $\eta = 0.9$), когда алгоритм перепрыгивает через минимум $x^* = 3$ на противоположную сторону параболы с затухающей амплитудой.
3. **Что происходит при слишком большом шаге?**  
   При $\eta = 1.1$ (вообще при $\eta > 1.0$) амплитуда колебаний растёт на каждом шаге — алгоритм расходится ($f(x_t) \to +\infty$).
4. **Почему слишком маленький шаг тоже может быть нежелателен?**  
   При $\eta = 0.01$ движение к минимуму слишком медленное: за 25 итераций алгоритм не успевает приблизиться к $x^* = 3$.
5. **Чем градиентный спуск принципиально отличается от методов одномерного поиска из части A?**  
   Методы части A — **безградиентные** (0-го порядка): они опираются только на значения функции и требуют исходного отрезка локализации $[a, b]$. Градиентный спуск — метод **1-го порядка**: он использует производную $f'(x)$ для определения направления и величины шага и легко обобщается на многомерные пространства.

---
## Часть C. Применение в машинном обучении

### Задание C1. Генерация данных

In [ ]:
np.random.seed(42)
n_samples = 150
X = np.random.uniform(-5, 5, size=n_samples)
noise = np.random.normal(0, 1.0, size=n_samples)
y = 2.0 * X + 1.0 + noise

plt.figure(figsize=(8, 4.5))
plt.scatter(X, y, alpha=0.65, label='Синтетические данные')
plt.xlabel('x')
plt.ylabel('y')
plt.title('Синтетическая выборка: y = 2x + 1 + ε (n = 150)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()

### Задание C2. Линейная регрессия и градиенты MSE
Модель: $\hat{y}_i = w x_i + b$. Функция потерь:
$$L(w, b) = \frac{1}{n}\sum_{i=1}^{n}(w x_i + b - y_i)^2.$$
Частные производные (градиенты):
$$\frac{\partial L}{\partial w} = \frac{2}{n}\sum_{i=1}^{n}(w x_i + b - y_i)x_i, \qquad \frac{\partial L}{\partial b} = \frac{2}{n}\sum_{i=1}^{n}(w x_i + b - y_i).$$

### Задание C3. Обучение модели с помощью градиентного спуска

In [ ]:
def compute_mse(w, b, x, y):
    return np.mean((w * x + b - y) ** 2)

def compute_gradients(w, b, x, y):
    error = (w * x + b) - y
    dw = (2.0 / len(x)) * np.sum(error * x)
    db = (2.0 / len(x)) * np.sum(error)
    return dw, db

w_cur, b_cur = 0.0, 0.0
lr_reg = 0.05
epochs = 150

w_hist = [w_cur]
b_hist = [b_cur]
loss_hist = [compute_mse(w_cur, b_cur, X, y)]

for _ in range(epochs):
    dw, db = compute_gradients(w_cur, b_cur, X, y)
    w_cur -= lr_reg * dw
    b_cur -= lr_reg * db
    w_hist.append(w_cur)
    b_hist.append(b_cur)
    loss_hist.append(compute_mse(w_cur, b_cur, X, y))

print(f"Найденные параметры: w = {w_cur:.4f} (истинное 2.0), b = {b_cur:.4f} (истинное 1.0)")
print(f"Финальный MSE: {loss_hist[-1]:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(loss_hist, color='crimson', lw=2)
axes[0].set_xlabel('Эпоха')
axes[0].set_ylabel('MSE')
axes[0].set_title('Изменение функции потерь при обучении')
axes[0].grid(True, linestyle='--', alpha=0.5)

axes[1].scatter(X, y, alpha=0.5, label='Данные с шумом')
x_line = np.linspace(X.min(), X.max(), 200)
axes[1].plot(x_line, w_cur * x_line + b_cur, color='black', lw=2, label=f'Обученная модель: {w_cur:.2f}x + {b_cur:.2f}')
axes[1].plot(x_line, 2.0 * x_line + 1.0, '--', color='blue', lw=1.5, label='Истинная прямая: 2x + 1')
axes[1].set_xlabel('x')
axes[1].set_ylabel('y')
axes[1].set_title('Аппроксимация данных линейной регрессией')
axes[1].grid(True, linestyle='--', alpha=0.5)
axes[1].legend()

plt.tight_layout()
plt.show()

#### Ответы на вопросы (Часть C)
1. **Что здесь является параметрами модели?** Коэффициент наклона $w$ и свободный член $b$.
2. **Что является целевой функцией?** Среднеквадратичная ошибка (MSE) $L(w, b) = \frac{1}{n}\sum_{i=1}^n (w x_i + b - y_i)^2$.
3. **Что означает обучение модели с точки зрения оптимизации?** Итеративный подбор параметров $(w, b)$, минимизирующих функцию потерь $L(w, b)$ с помощью градиентного спуска.
4. **Почему найденные $w$ и $b$ могут не совпасть точно с $2$ и $1$?** Из-за случайного шума $\varepsilon \sim \mathcal{N}(0, 1)$ и конечного объёма выборки ($n = 150$) минимум эмпирического риска слегка смещён относительно истинных параметров генерации данных.

---
## Бонусная часть (до +2 баллов)


### Бонус 2. Плохо обусловленная функция $f(x, y) = x^2 + 100y^2$

In [ ]:
def ravine_f(x, y):
    return x ** 2 + 100.0 * (y ** 2)

def ravine_grad(x, y):
    return np.array([2.0 * x, 200.0 * y])

def gd_2d(start_point, lr, n_steps=35):
    point = np.array(start_point, dtype=float)
    traj = [point.copy()]
    for _ in range(n_steps):
        point = point - lr * ravine_grad(point[0], point[1])
        traj.append(point.copy())
        if np.linalg.norm(point) > 1e4:
            break
    return np.array(traj)

start = [-8.0, 1.8]
lrs_2d = [0.002, 0.0095]
xg = np.linspace(-10, 10, 400)
yg = np.linspace(-2.5, 2.5, 400)
Xg, Yg = np.meshgrid(xg, yg)
Zg = ravine_f(Xg, Yg)
levels = np.logspace(-1, 3.5, 20)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
for ax, lr in zip(axes, lrs_2d):
    traj = gd_2d(start, lr=lr, n_steps=35)
    ax.contour(Xg, Yg, Zg, levels=levels, cmap='viridis', alpha=0.65)
    ax.plot(traj[:, 0], traj[:, 1], 'r.-', lw=1.5, markersize=6, label=f'GD (η = {lr})')
    ax.scatter([0], [0], color='gold', marker='*', s=150, zorder=5, label='Минимум (0, 0)')
    ax.scatter([start[0]], [start[1]], color='blue', s=50, zorder=5, label='Старт')
    ax.set_title(f'Траектория при η = {lr}')
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.legend(loc='upper right')

plt.suptitle(r'Градиентный спуск на овражной функции $f(x, y) = x^2 + 100y^2$', fontsize=13)
plt.tight_layout()
plt.show()

**Объяснение:** линии уровня вытянуты вдоль оси $x$ (отношение кривизн $100:1$). Градиент в основном направлен поперёк «оврага» (по оси $y$). При $\eta = 0.0095$ шаг близок к пределу устойчивости по $y$ ($1/100 = 0.01$), поэтому возникают сильные зигзагообразные осцилляции вдоль крутых стенок, а вдоль оси $x$ алгоритм продвигается очень медленно.

---
## Общий вывод

В ходе лабораторной работы реализованы и исследованы три метода одномерной оптимизации (дихотомия, золотое сечение, метод Фибоначчи) для функции $f(x) = x + \frac{1}{x - 2.5}$ на $[3, 6]$, а также градиентный спуск на модельной функции и в задаче обучения линейной регрессии.
- Среди безградиентных методов золотое сечение и метод Фибоначчи экономнее дихотомии по числу вычислений $f(x)$, а при одинаковом бюджете метод Фибоначчи обеспечивает наименьший интервал неопределённости.
- Градиентный спуск использует информацию о производной и переносится на многомерные задачи ML, но критически зависит от выбора шага обучения $\eta$ и обусловленности функции потерь.